# 2. Count and presence models: the national picture

**Question.** After accounting for population, are outlets more or less common per person in poorer places than richer ones, and does that hold among places with the same amount of shopping?

**Model.** A negative binomial (NB2) count model with estimated dispersion:

- **Outcome and offset.** The outcome is the number of outlets in a tile, and the offset is the log of its people, so the model compares outlets per person.
- **Intercepts.** One per province, which absorbs PCSO branch practice and regional differences.
- **Wealth.** Wealth enters as the RWI in units of one national tile standard deviation. The exponentiated coefficient is the **wealth rate ratio**: how many times as many outlets per person a tile one SD richer has, with the other terms held fixed.

Above 1 means more outlets per person in richer places, below 1 in poorer places.
It is an association between wealth and outlet density; it does not say why outlets are where they are.

**Variants.**

- Wealth only.
- With urban share, which is the main model without shops.
- With shops, from OpenStreetMap and Overture, which is the main model with shops.
- With shops and road length.

Commerce may sit between wealth and outlets as much as beside them, so both main versions are reported.

**Presence model.** A logit of whether a tile has any outlet, with log population as a covariate. Its exponentiated coefficient is an odds ratio.

**Spatial checks.** Moran's I is computed on mid-quantile and deviance residuals of each main model, with neighbours being the 8 tiles around each tile.
Where residuals cluster, the model is refitted with Moran eigenvector spatial filters, a frequentist stand-in for a BYM2 model.

**Errors.** Clustered by province for tiles, and by RWI tile for barangays.

In [1]:
import sys
from pathlib import Path

# The notebooks import their logic from src/: the shared helpers and the analysis functions.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir())
for folder in ("src/shared", "src/03_analyse"):
    sys.path.insert(0, str(ROOT / folder))

import pandas as pd
from IPython.display import Image

from analysis_samples import RWI_SD, TABLES

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

import count_models as cm

print(f"Wealth unit: 1 national tile SD = {RWI_SD:.4f} RWI points")

Wealth unit: 1 national tile SD = 0.4937 RWI points


## National tiles (the main unit)

In [2]:
national_tiles = cm.fit_scope("national_tiles")

National tiles: 40,737 units, 7,271 outlets, 2,330 units with one
  count wealth only        rate ratio 2.450 (2.233-2.688)
  count + urban share      rate ratio 2.132 (1.963-2.317)
  spatial refit (10 eigenvectors) rate ratio 2.139 (1.969-2.323); residual Moran's I -0.0117 -> -0.0131
  count + shops            rate ratio 1.173 (1.070-1.286)
  spatial refit (10 eigenvectors) rate ratio 1.176 (1.072-1.290); residual Moran's I 0.0024 -> 0.0008
  count + shops and roads  rate ratio 1.240 (1.132-1.358)
  presence + urban share   odds ratio 2.176 (1.921-2.465) [497 units in areas with no variation dropped; 18 fitted at 0 or 1]
  presence + shops         odds ratio 1.591 (1.379-1.836) [497 units in areas with no variation dropped; 0 fitted at 0 or 1]
  LISA clusters: {'Not significant': 36854, 'Low-High': 1722, 'Low-Low': 1363, 'High-Low': 372, 'High-High': 222, 'No neighbours': 204}


In [3]:
cm.summary(national_tiles)

,scope,unit,model,variant,estimate,n,wealth_ratio,ci_low,ci_high,p_value
0,National,tile,count,wealth only,rate ratio,40737,2.450,2.233,2.688,0.0000
1,National,tile,count,+ urban share,rate ratio,40737,2.132,1.963,2.317,0.0000
2,National,tile,"count, spatial refit",+ urban share,rate ratio,40737,2.139,1.969,2.323,0.0000
3,National,tile,count,+ shops,rate ratio,40737,1.173,1.070,1.286,0.0007
4,National,tile,"count, spatial refit",+ shops,rate ratio,40737,1.176,1.072,1.290,0.0006
5,National,tile,count,+ shops and roads,rate ratio,40737,1.240,1.132,1.358,0.0000
6,National,tile,presence,+ urban share,odds ratio,40240,2.176,1.921,2.465,0.0000
7,National,tile,presence,+ shops,odds ratio,40240,1.591,1.379,1.836,0.0000


**Reading.** Holding urban share and the province fixed, a tile one SD richer has about 2.1 times as many outlets per person (95% CI 1.96-2.32).
Adding shops shrinks the ratio to 1.17 (1.07-1.29): much of the wealth gradient goes with where shops are, but richer tiles still have more outlets per person than equally commercial poorer ones.
Adding road length leaves it at 1.24.
The presence model says the same about having any outlet at all: an odds ratio of 2.18 without shops and 1.59 with them.

In [4]:
pd.DataFrame(national_tiles["spatial"])[["model", "variant", "residuals", "morans_i", "z", "p_value"]].round(4)

,model,variant,residuals,morans_i,z,p_value
0,count,+ urban share,mid-quantile,-0.0117,-4.1610,0.001
1,count,+ urban share,deviance,0.0212,7.5257,0.001
2,"count, spatial refit",+ urban share,mid-quantile,-0.0131,-4.6277,0.001
3,count,+ shops,mid-quantile,0.0024,0.8322,0.424
4,count,+ shops,deviance,0.0218,7.7352,0.001
5,"count, spatial refit",+ shops,mid-quantile,0.0008,0.2839,0.776
6,presence,+ urban share,mid-quantile,-0.0095,-3.1394,0.003


**Reading.** Tile residuals barely cluster: Moran's I is about zero or slightly negative on mid-quantile residuals, and 0.02 on deviance residuals.
The deviance figure was enough to trigger the spatial refit, which needed only 10 eigenvectors and leaves the wealth ratio where it was (2.14 and 1.18).
The local clusters (LISA) of these residuals are the hot-spot layer of the web map.

## National barangays (the check)

In [5]:
national_barangays = cm.fit_scope("national_barangays")

National barangays: 41,923 units, 7,282 outlets, 3,884 units with one
  count wealth only        rate ratio 2.820 (2.672-2.977)
  count + urban share      rate ratio 2.362 (2.216-2.518)
  spatial refit (100 eigenvectors) rate ratio 2.416 (2.271-2.570); residual Moran's I 0.0391 -> 0.0343
  count + shops            rate ratio 1.496 (1.393-1.607)
  spatial refit (100 eigenvectors) rate ratio 1.508 (1.405-1.618); residual Moran's I 0.0419 -> 0.0375
  count + shops and roads  rate ratio 1.221 (1.143-1.304)
  presence + urban share   odds ratio 2.633 (2.465-2.813) [434 units in areas with no variation dropped; 0 fitted at 0 or 1]
  presence + shops         odds ratio 1.411 (1.322-1.506) [434 units in areas with no variation dropped; 0 fitted at 0 or 1]


In [6]:
cm.summary(national_barangays)

,scope,unit,model,variant,estimate,n,wealth_ratio,ci_low,ci_high,p_value
0,National,barangay,count,wealth only,rate ratio,41923,2.820,2.672,2.977,0.0
1,National,barangay,count,+ urban share,rate ratio,41923,2.362,2.216,2.518,0.0
2,National,barangay,"count, spatial refit",+ urban share,rate ratio,41923,2.416,2.271,2.570,0.0
3,National,barangay,count,+ shops,rate ratio,41923,1.496,1.393,1.607,0.0
4,National,barangay,"count, spatial refit",+ shops,rate ratio,41923,1.508,1.405,1.618,0.0
5,National,barangay,count,+ shops and roads,rate ratio,41923,1.221,1.143,1.304,0.0
6,National,barangay,presence,+ urban share,odds ratio,41489,2.633,2.465,2.813,0.0
7,National,barangay,presence,+ shops,odds ratio,41489,1.411,1.322,1.506,0.0


**Reading.** By barangay the gradient is the same shape and a little steeper: 2.36 without shops and 1.50 with them.
Residuals cluster more here (Moran's I about 0.04), because neighbouring barangays often share one wealth tile, and the spatial refit leaves the estimates unchanged (2.42 and 1.51).

## Sensitivities: how outlets are placed

These refit the two main tile models with the outcome counted two other ways:

- only GPS points that agree with the town in their address;
- adding the 133 outlets that could be placed only at a town centre.

In [7]:
sensitivities = cm.fit_sensitivities()
cm.summary(sensitivities)

  GPS points consistent with their address only, + urban share: rate ratio 2.122 (1.944-2.317)
  GPS points consistent with their address only, + shops: rate ratio 1.155 (1.053-1.268)
  adding outlets placed only at a town centre, + urban share: rate ratio 2.110 (1.947-2.288)
  adding outlets placed only at a town centre, + shops: rate ratio 1.166 (1.066-1.275)


,scope,unit,model,variant,estimate,n,wealth_ratio,ci_low,ci_high,p_value
0,National,tile,"count, sensitivity: GPS points consistent with...",+ urban share,rate ratio,40737,2.122,1.944,2.317,0.0000
1,National,tile,"count, sensitivity: GPS points consistent with...",+ shops,rate ratio,40737,1.155,1.053,1.268,0.0023
2,National,tile,"count, sensitivity: adding outlets placed only...",+ urban share,rate ratio,40737,2.110,1.947,2.288,0.0000
3,National,tile,"count, sensitivity: adding outlets placed only...",+ shops,rate ratio,40737,1.166,1.066,1.275,0.0008


**Reading.** Neither placement choice moves the ratios: 2.11 to 2.12 without shops and 1.16 to 1.17 with them.

The model rows are saved for notebook 3, which adds NCR and writes `results/tables/models.csv`, `coefficients.csv` and `spatial_checks.csv`.

In [8]:
cm.save_part(national_tiles, "national_tiles")
cm.save_part(national_barangays, "national_barangays")
cm.save_part(sensitivities, "sensitivities")